# YOLO 2회차 추가 실습 — 말로 찾는 탐지기 (YOLOE)

실습 A에서는 상자를 찾기 위해 라벨이 달린 이미지 1,920장으로 모델을 학습했습니다. 그런데 **학습 없이, 찾고 싶은 물체의 이름만 말해서** 찾을 수는 없을까요?
YOLOE는 글자(텍스트 프롬프트)로 찾을 대상을 지정하는 **오픈 보캐뷸러리(open-vocabulary) 탐지 모델**입니다. 이 실습에서는 YOLOE를 써 보고, 어떤 단어로 부르느냐에 따라 성능이 얼마나 달라지는지 **프롬프트 대결**을 하고, 직접 학습한 모델과도 맞붙여 봅니다.

| 단계 | 만드는 것 | 결과물 |
|---|---|---|
| Step 1 | 학습 없이 말로 탐지하기 | 탐지 결과 이미지 |
| Step 2 | 같은 사진, 다른 단어 — 프롬프트에 따라 달라지는 결과 | 비교 그림 |
| Step 3 | 프롬프트 대결 — 단어별 mAP(평균 정밀도 — 0~1 사이 탐지 성능 점수) 측정 | `prompt_battle.csv` |
| Step 4 | 직접 학습한 모델과 대결 — 정확도·속도·준비 비용 | 비교 표 |
| Step 5 | 응용: 안전모 미착용자 찾기 + 정답 라벨로 채점 | `helmet_check.csv` |

**데이터셋** (Ultralytics 공개 데이터셋, 자동 다운로드)
- *Package Segmentation*(세그멘테이션 — 물체 윤곽을 픽셀 단위로 칠하는 작업) — 실습 A와 같은 택배 상자 CCTV 이미지 (검증 188장)
- *Construction-PPE* — 건설 현장 작업자 안전장비 이미지 (학습 1,132장 · 검증 143장 · 테스트 141장, 안전모·조끼·장갑 등 11개 클래스)

**실행 환경**: Google Colab, 런타임 유형 **T4 GPU**(그래픽 처리 장치 — 딥러닝 계산을 빠르게 하는 칩)

## 0. 준비

In [ ]:
# !로 시작하는 줄은 파이썬이 아닌 셸 명령
# YOLO·YOLOE 모델과 학습·평가 도구 설치 (-q: 설치 메시지 줄임)
!pip -q install ultralytics
# YOLOE가 단어를 벡터로 바꿀 때 쓰는 CLIP 패키지 설치
!pip -q install git+https://github.com/ultralytics/CLIP.git

In [ ]:
# 필요한 라이브러리 불러오기
import os, json, time, shutil, glob
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import display
from ultralytics import YOLO, YOLOE  # YOLO: 일반 모델(실습 A용), YOLOE: 단어로 찾는 모델
from ultralytics.data.utils import check_det_dataset  # 데이터셋 내려받기 + 경로·클래스 정보 반환

print('GPU 사용 가능:', torch.cuda.is_available())  # T4 GPU 런타임이면 True

# Colab에서만 성공하는 import로 실행 환경 구분
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

USE_DRIVE = True  # False: 로컬 outputs 사용 (Step 4에서 실습 A 모델 못 찾음)
if IN_COLAB and USE_DRIVE:
    drive.mount('/content/drive')  # Google 계정 권한 창이 뜨면 허용
    DAY2_DIR = Path('/content/drive/MyDrive/yolo_day2')  # 실습 A와 같은 2회차 공용 폴더
else:
    DAY2_DIR = Path('outputs')
SAVE_DIR = DAY2_DIR / 'prompt'  # 이 실습의 결과 폴더
SAVE_DIR.mkdir(parents=True, exist_ok=True)  # 상위 폴더까지 생성, 이미 있어도 오류 없음
print('결과 저장 위치:', SAVE_DIR.resolve())

# OpenCV BGR 이미지 한 장을 화면에 표시
def show_bgr(img, title=None, size=(9, 6)):
    plt.figure(figsize=size)
    plt.imshow(img[..., ::-1]); plt.axis('off')  # [..., ::-1]: 채널 순서 BGR → RGB로 뒤집기
    if title:
        plt.title(title)
    plt.show()

# 이미지 여러 장을 cols열 격자로 표시 (titles: 칸별 제목)
def show_grid(imgs, titles=None, cols=3, size=4):
    rows = (len(imgs) + cols - 1) // cols  # 필요한 행 수 (올림 나눗셈)
    fig, axes = plt.subplots(rows, cols, figsize=(size * cols, size * rows))
    for ax, i in zip(np.array(axes).ravel(), range(rows * cols)):  # 2차원 칸 배열을 한 줄로 펴서 차례로 채움
        ax.axis('off')  # 축 눈금 숨김 (남는 칸은 빈칸으로)
        if i < len(imgs):
            ax.imshow(imgs[i][..., ::-1])  # BGR → RGB로 바꿔 표시
            if titles:
                ax.set_title(titles[i], fontsize=9)
    plt.tight_layout(); plt.show()

# 결과 파일을 SAVE_DIR(Drive)로 복사하고 복사된 경로 반환
def save_out(path):
    dst = SAVE_DIR / Path(path).name
    if Path(path).resolve() != dst.resolve():  # 이미 같은 파일이면 복사 생략
        shutil.copy(path, dst)
    return dst

In [ ]:
# 데이터셋 두 개 내려받기 (처음 한 번, 합계 약 280MB)
PKG = check_det_dataset('package-seg.yaml')  # 택배 상자 CCTV (실습 A와 같은 데이터)
PPE = check_det_dataset('construction-ppe.yaml')  # 건설 현장 안전장비, 11개 클래스
PKG_ROOT, PPE_ROOT = Path(PKG['path']), Path(PPE['path'])  # 데이터셋 최상위 폴더 (images/, labels/ 포함)
print('package-seg     :', PKG_ROOT, PKG['names'])  # names: {클래스 번호: 이름} 딕셔너리
print('construction-ppe:', PPE_ROOT, PPE['names'])

---
## Step 1. 학습 없이 말로 탐지하기

### YOLOE는 어떻게 단어를 알아듣나

```
"cardboard box" ──▶ 텍스트 인코더(MobileCLIP) ──▶ 단어 벡터 ─┐
                                                              ├─▶ 비슷한 영역 = 탐지 결과
이미지 ───────────▶ YOLO 백본 ─────────────────▶ 영역 벡터 ──┘
```

- 텍스트 인코더(text encoder, 글자를 숫자로 바꾸는 신경망)가 단어를 숫자 벡터(숫자 여러 개를 줄 세운 목록)로 바꾸고, 이미지의 각 영역 벡터와 **얼마나 비슷한지** 비교합니다.
- 그래서 학습 때 클래스 목록에 없던 단어도 찾을 수 있습니다. 대신 단어를 고르는 방식에 따라 결과가 크게 달라집니다.
- `set_classes([...])`로 찾을 단어 목록을 정합니다. 처음 호출할 때 텍스트 인코더 가중치(weights, 학습으로 얻은 모델 내부 숫자 값)를 내려받아 시간이 조금 걸립니다.

In [ ]:
model = YOLOE('yoloe-26s-seg.pt')  # YOLOE s 크기 세그멘테이션 모델 (없으면 자동 다운로드)
model.set_classes(['person', 'helmet', 'safety vest', 'cardboard box'])  # 찾을 단어 목록 (첫 호출 때 텍스트 인코더 다운로드)

ppe_test = sorted((PPE_ROOT / 'images' / 'test').glob('*'))  # 안전장비 테스트 이미지 (정렬 → 매번 같은 순서)
pkg_val = sorted((PKG_ROOT / 'images' / 'val').glob('*.jpg'))  # 택배 CCTV 검증 이미지
samples = [ppe_test[0], ppe_test[2], ppe_test[5], pkg_val[3], pkg_val[95], pkg_val[150]]  # 공사 현장 3장 + CCTV 3장

# 샘플마다 탐지하고 결과 이미지·제목 모으기
imgs, titles = [], []
for p in samples:
    r = model.predict(str(p), conf=0.25, verbose=False)[0]  # conf=0.25: 점수 0.25 미만 박스는 버림, [0]: 첫 이미지 결과
    imgs.append(r.plot())  # 박스·마스크·라벨을 그린 이미지(BGR)
    names = [r.names[int(c)] for c in r.boxes.cls]  # 박스별 클래스 번호 → 단어
    titles.append(', '.join(f'{n} {names.count(n)}' for n in dict.fromkeys(names)) or 'nothing found')  # 단어별 개수 제목, 하나도 없으면 nothing found
show_grid(imgs, titles)

**확인해 보기** — 공사 현장 사진에서는 사람·안전모·조끼를 잘 찾지만, 택배 분류장 CCTV에서는 상자를 거의 찾지 못합니다. 왜 그럴까요?
YOLOE가 배운 인터넷 사진 속 '상자'는 정면에서 또렷하게 찍힌 모습이 많습니다. 위에서 내려다보는 흐릿한 CCTV 화면, 붉은 조명 아래의 상자는 낯선 모습입니다.

---
## Step 2. 같은 사진, 다른 단어

같은 CCTV 사진에 단어만 바꿔 넣고 결과를 비교합니다. 사람에게는 'package'와 'cardboard box'가 같은 물건이지만 모델에게는 다른 벡터입니다.

In [ ]:
PROMPTS = ['package', 'box', 'cardboard box', 'parcel', 'carton box']  # 비교할 단어 5개 (Step 3에서도 사용)
probe = pkg_val[60]  # 비교에 쓸 CCTV 사진 한 장
LOW_CONF = 0.003          # CCTV 상자는 점수가 매우 낮아 기준을 크게 낮춤 (올리면 박스 줄어듦)

# 단어를 하나씩 바꿔 가며 같은 사진 탐지
imgs, titles = [], []
for word in PROMPTS:
    model.set_classes([word])  # 찾을 단어만 교체 (모델 가중치는 그대로)
    r = model.predict(str(probe), conf=LOW_CONF, verbose=False)[0]
    conf = r.boxes.conf.cpu().numpy()  # 박스별 점수 → 넘파이 배열 (GPU → CPU)
    imgs.append(r.plot(labels=False))  # 클래스 이름은 빼고 박스·점수만 그림
    titles.append(f'"{word}": {len(conf)} boxes, max conf {conf.max():.3f}' if len(conf) else f'"{word}": 0 boxes')  # 박스가 0개면 max() 오류를 피해 따로 표기
# 이미지 경로 → 정답 라벨 경로 (images→labels, 확장자→.txt)
gt_n = len(open(str(probe).replace('images', 'labels').rsplit('.', 1)[0] + '.txt').read().strip().splitlines())  # 라벨 파일 줄 수 = 정답 상자 수
show_grid(imgs, titles)
print('이 사진의 정답 상자 수:', gt_n)

두 가지를 확인하세요.
- 단어마다 찾은 상자 수가 다릅니다. 'package'는 거의 못 찾고 'cardboard box'나 'parcel'은 여러 개를 찾습니다.
- **점수(conf, 신뢰도 — 모델이 박스마다 매기는 0~1 확신 점수)가 0.1 안팎으로 매우 낮습니다.** 낯선 장면에서 제로샷(zero-shot, 이 데이터로 추가 학습 없이 바로 쓰는) 모델은 정답을 맞혀도 자신 없는 점수를 줍니다. 그래서 기본 기준(0.25)으로는 아무것도 안 보이고, 기준을 아주 낮춰야 보입니다. 실제로 쓰려면 이 기준도 데이터로 정해야 합니다.

한 장만 보고 단어를 고르면 우연에 휘둘립니다. Step 3에서는 **검증 이미지 188장 전체**로 단어별 성적을 매깁니다.

---
## Step 3. 프롬프트 대결 — 단어별 mAP

YOLOE의 `val()`(검증 데이터로 성능을 재는 평가 함수)은 데이터셋 설정 파일의 **클래스 이름을 그대로 프롬프트로** 씁니다. 그래서 이름만 바꾼 설정 파일을 단어마다 만들고 평가하면 됩니다.
이미지와 정답 라벨은 같고 **단어만 다르므로**, 점수 차이는 순전히 단어 선택 때문입니다.

In [ ]:
# 단어(word)를 클래스 이름으로 넣은 데이터셋 설정 파일을 만들고 경로 반환
def make_yaml(word, path):
    cfg = {'path': str(PKG_ROOT), 'train': 'images/train', 'val': 'images/val', 'names': {0: word}}  # 이미지·라벨은 그대로, 클래스 0 이름만 단어로
    Path(path).write_text(json.dumps(cfg))     # JSON은 YAML의 부분집합이라 그대로 읽힘
    return path

# 단어마다 설정 파일을 만들어 검증 이미지 188장으로 평가
battle = []
for word in PROMPTS:
    yml = make_yaml(word, f'pkg_{word.replace(" ", "_")}.yaml')  # 공백은 밑줄로 바꿔 파일 이름 만듦
    m = YOLOE('yoloe-26s-seg.pt')  # 단어마다 새 모델 — 이전 설정이 남지 않게
    # batch=16: 한 번에 처리할 이미지 수 (메모리 부족하면 8로)
    res = m.val(data=yml, split='val', imgsz=640, batch=16, verbose=False, plots=False)  # val(): 설정 파일의 클래스 이름을 프롬프트로 써서 평가
    battle.append({'prompt': word, 'precision': round(res.box.mp, 3), 'recall': round(res.box.mr, 3),  # mp: 평균 정밀도, mr: 평균 재현율
                   'map50': round(res.box.map50, 3), 'map50_95': round(res.box.map, 3)})  # res.box.map = mAP50-95
    print(f'{word:15s} mAP50 {res.box.map50:.3f}')

battle = pd.DataFrame(battle).sort_values('map50', ascending=False).reset_index(drop=True)  # mAP50 높은 순 정렬, 번호 0부터 다시 → 0번이 1등
battle.to_csv('prompt_battle.csv', index=False); save_out('prompt_battle.csv')  # CSV 저장 후 Drive로 복사
battle

In [ ]:
plt.figure(figsize=(8, 3.5))
bars = plt.bar(battle['prompt'], battle['map50'], color=['tab:green'] + ['tab:gray'] * (len(battle) - 1))  # 1등(첫 막대)만 초록, 나머지 회색
for b, v in zip(bars, battle['map50']):
    plt.text(b.get_x() + b.get_width() / 2, v + 0.01, f'{v:.2f}', ha='center')  # 막대 가운데 위에 값 표시
plt.ylim(0, 1); plt.ylabel('mAP50'); plt.title('Prompt battle on package-seg val (188 images)')  # y축 0~1 고정 → 단어 사이 차이를 공정하게 비교
plt.show()
BEST_PROMPT = battle.loc[0, 'prompt']  # 1등 단어 → Step 4에서 사용
print('우승 단어:', BEST_PROMPT)

> mAP는 점수의 **순서**(정답 박스가 헛박스보다 높은 점수를 받는가)로 계산하므로, 점수가 0.01처럼 낮아도 순서만 잘 맞으면 높게 나옵니다. 그래서 mAP가 괜찮아도 실제 서비스에서는 Step 2에서 본 것처럼 낮은 기준을 따로 정해야 합니다.

**확인해 보기**
- 1등 단어와 꼴찌 단어의 mAP50(IoU 0.5 기준 mAP — IoU는 겹친 넓이 ÷ 합친 넓이, 0.5 이상이면 맞힌 것으로 침) 차이는 얼마인가요? 모델을 바꾸지 않고 **단어 하나로** 생긴 차이입니다.
- 정밀도(precision, 찾은 것 중 맞은 비율)와 재현율(recall, 실제 정답 중 찾아낸 비율)을 보세요. 어떤 단어는 많이 찾지만 헛탐지가 많고, 어떤 단어는 적게 찾지만 정확합니다.
- 나만의 단어를 `PROMPTS`에 추가해 순위를 바꿔 보세요. 예: `'brown box'`, `'shipping box'`, `'box on conveyor'`

---
## Step 4. 직접 학습한 모델과 대결

실습 A에서 학습해 Drive에 저장한 `box_seg_best.pt`가 있으면 불러와 **같은 검증 이미지**로 평가합니다. 정확도뿐 아니라 **이미지 한 장 처리 시간**과 **준비 비용**(라벨 이미지 수, 학습 시간)도 함께 비교합니다.

In [ ]:
# 모델 m으로 images 앞 n장을 예측해 한 장당 평균 시간(ms) 반환
def ms_per_image(m, images, n=40, **kw):
    m.predict(str(images[0]), verbose=False, **kw)              # 예열 (첫 호출은 느림)
    t0 = time.time()
    for p in images[:n]:
        m.predict(str(p), verbose=False, **kw)
    return (time.time() - t0) / n * 1000  # 초 → 밀리초

# ① 우승 단어를 쓴 YOLOE 제로샷 모델 (라벨 0장, 학습 없음)
compare = []
zs = YOLOE('yoloe-26s-seg.pt'); zs.set_classes([BEST_PROMPT])  # 우승 단어 하나만 찾게 설정
compare.append({'model': f'YOLOE zero-shot ("{BEST_PROMPT}")', 'labeled_images': 0, 'training': 'none',
                'map50': battle.loc[0, 'map50'], 'ms_per_image': round(ms_per_image(zs, pkg_val), 1)})  # mAP50은 Step 3에서 잰 1등 값 재사용

# ② 실습 A에서 학습한 모델 (Drive에 파일이 있을 때만)
LAB_A_MODEL = DAY2_DIR / 'box_seg_best.pt'  # MyDrive/yolo_day2/box_seg_best.pt
if LAB_A_MODEL.exists():
    ft = YOLO(LAB_A_MODEL)
    res = ft.val(data='package-seg.yaml', split='val', batch=16, verbose=False, plots=False)  # 같은 검증 이미지 188장으로 평가
    compare.append({'model': 'Lab A fine-tuned (yolo26n-seg)', 'labeled_images': 1920, 'training': 'yes',
                    'map50': round(res.box.map50, 3), 'ms_per_image': round(ms_per_image(ft, pkg_val), 1)})
else:
    print('실습 A 모델이 없습니다. 실습 A를 먼저 실행하면 이 표에 함께 나옵니다.')

compare = pd.DataFrame(compare)
compare.to_csv('model_compare.csv', index=False); save_out('model_compare.csv')  # CSV 저장 후 Drive로 복사
compare

**생각해 보기 — 언제 무엇을 쓸까**

| 상황 | 알맞은 선택 |
|---|---|
| 새 물체를 **오늘 당장** 찾아봐야 함, 라벨 없음 | YOLOE 제로샷 + 좋은 단어 찾기 |
| 정확도가 서비스 품질을 좌우함 (카운팅·과금·안전) | 라벨을 모아 직접 학습 |
| 라벨링 비용을 줄이고 싶음 | YOLOE로 **라벨 초안**을 만들고 사람이 고쳐 학습 → *오토 라벨링 공장* 실습 |

YOLOE의 `s`(small) 모델은 학습한 `n`(nano, 가장 작은 크기) 모델보다 크기 때문에 느릴 수 있습니다. 엣지 장비(edge device, 현장에 두고 직접 추론하는 소형 컴퓨터 — 3회차 Jetson)에서는 속도 차이가 더 중요해집니다.

---
## Step 5. 응용 — 안전모 미착용자 찾기

YOLOE로 `person`과 `helmet`을 찾은 뒤, **규칙**을 붙여 '안전모를 쓰지 않은 사람'을 찾습니다.

```
사람 박스의 위쪽 35% (머리 영역) 안에 안전모 박스의 중심이 있는가?
   예  → 착용 (초록)
 아니요 → 미착용 (빨강)
```

Construction-PPE 데이터셋에는 사람이 직접 단 `no_helmet`(안전모 미착용) 라벨이 있습니다. 이 라벨로 우리 규칙을 **채점**합니다.

In [ ]:
HEAD_RATIO = 0.35  # 사람 박스 위쪽 35%를 머리 영역으로 봄 (키우면 착용 판정 늘어남)

# 사람마다 머리 영역에 안전모 중심이 있는지 판정
def helmet_check(r):
    """YOLOE 결과 → [(사람 박스, 착용 여부)]"""
    boxes, cls = r.boxes.xyxy.cpu().numpy(), r.boxes.cls.cpu().numpy().astype(int)  # xyxy: 꼭짓점 픽셀 좌표, cls: 클래스 번호
    names = r.names  # {클래스 번호: 단어} 딕셔너리
    persons = boxes[[names[c] == 'person' for c in cls]]  # True/False 목록으로 person 박스만 골라냄
    helmets = boxes[[names[c] == 'helmet' for c in cls]]  # helmet 박스만 골라냄
    out = []
    for x1, y1, x2, y2 in persons:
        head_y2 = y1 + (y2 - y1) * HEAD_RATIO  # 머리 영역 아래 경계 y (이미지 y는 아래로 커짐)
        # 안전모 중심 x가 사람 박스 안 + 중심 y가 머리 영역 안이면 착용
        # (위쪽은 사람 박스 높이의 10%만큼 더 위까지 허용)
        worn = any(x1 <= (hx1 + hx2) / 2 <= x2 and y1 - (y2 - y1) * 0.1 <= (hy1 + hy2) / 2 <= head_y2
                   for hx1, hy1, hx2, hy2 in helmets)  # 안전모를 하나씩 검사, 하나라도 맞으면 True
        out.append(((x1, y1, x2, y2), worn))
    return out

# 판정 결과를 이미지에 그림: 착용 초록, 미착용 빨강
def draw_check(img, checks):
    for (x1, y1, x2, y2), worn in checks:
        col = (0, 200, 0) if worn else (0, 0, 255)  # OpenCV 색 순서는 BGR → (0, 0, 255)가 빨강
        cv2.rectangle(img, (int(x1), int(y1)), (int(x2), int(y2)), col, 3)  # 사람 박스 테두리 (두께 3)
        label = 'HELMET OK' if worn else 'NO HELMET'
        cv2.rectangle(img, (int(x1), int(y1) - 24), (int(x1) + 130, int(y1)), col, -1)  # 글자 배경 띠 (두께 -1 = 속 채움)
        cv2.putText(img, label, (int(x1) + 4, int(y1) - 6), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)  # 흰 글자로 판정 표시
    return img

model.set_classes(['person', 'helmet'])  # Step 2에서 바꾼 단어 목록을 사람·안전모로 재설정
imgs, titles = [], []
for p in ppe_test[:6]:
    r = model.predict(str(p), conf=0.3, verbose=False)[0]  # 점수 0.3 이상 박스만 사용
    ch = helmet_check(r)
    imgs.append(draw_check(r.orig_img.copy(), ch))  # 원본 복사본에 그림 (원본 보존)
    titles.append(f'{sum(not w for _, w in ch)} without helmet / {len(ch)} people')  # 미착용 수 / 사람 수
show_grid(imgs, titles)

### 정답 라벨로 채점하기

이미지 단위로 채점합니다. 정답 라벨에 `no_helmet`이 하나라도 있으면 '미착용자 있음(양성)'입니다.

| | 규칙: 미착용자 있음 | 규칙: 없음 |
|---|---|---|
| **정답: 있음** | TP (잘 찾음) | FN (놓침 — 위험) |
| **정답: 없음** | FP (헛경보) | TN |

In [ ]:
NO_HELMET = [k for k, v in PPE['names'].items() if v == 'no_helmet'][0]  # 이름이 no_helmet인 클래스 번호 찾기

# 테스트 이미지마다 정답 라벨과 규칙 판정을 기록
rows = []
for p in ppe_test:
    lbl = PPE_ROOT / 'labels' / 'test' / (p.stem + '.txt')  # 이미지와 같은 이름의 정답 라벨 파일
    gt_cls = [int(l.split()[0]) for l in lbl.read_text().strip().splitlines()] if lbl.exists() else []  # 각 줄 첫 숫자 = 클래스 번호, 파일 없으면 빈 목록
    r = model.predict(str(p), conf=0.3, verbose=False)[0]
    ch = helmet_check(r)
    rows.append({'image': p.name, 'gt_no_helmet': int(NO_HELMET in gt_cls),  # 정답에 no_helmet이 있으면 1
                 'pred_no_helmet': int(any(not w for _, w in ch)), 'people_found': len(ch)})  # 규칙이 미착용자를 1명이라도 찾으면 1
check = pd.DataFrame(rows)
check.to_csv('helmet_check.csv', index=False); save_out('helmet_check.csv')  # CSV 저장 후 Drive로 복사

# 두 조건을 &로 묶어 TP·FN·FP·TN 이미지 수 세기
tp = int(((check.gt_no_helmet == 1) & (check.pred_no_helmet == 1)).sum())
fn = int(((check.gt_no_helmet == 1) & (check.pred_no_helmet == 0)).sum())
fp = int(((check.gt_no_helmet == 0) & (check.pred_no_helmet == 1)).sum())
tn = int(((check.gt_no_helmet == 0) & (check.pred_no_helmet == 0)).sum())
print(f'테스트 이미지 {len(check)}장 | TP {tp}  FN {fn}  FP {fp}  TN {tn}')
print(f'재현율(미착용 이미지를 찾은 비율) {tp / max(tp + fn, 1):.2f} | 정밀도(경보가 맞은 비율) {tp / max(tp + fp, 1):.2f}')  # max(..., 1): 분모 0 방지

In [ ]:
# 놓친 사례(FN)와 헛경보(FP) 눈으로 보기
imgs, titles = [], []
for kind, cond in [('FN (missed)', (check.gt_no_helmet == 1) & (check.pred_no_helmet == 0)),  # cond: 판다스 True/False 열 (FN, FP 차례로)
                   ('FP (false alarm)', (check.gt_no_helmet == 0) & (check.pred_no_helmet == 1))]:
    for name in check.loc[cond, 'image'].head(3):  # 조건에 맞는 이미지 이름 최대 3개
        r = model.predict(str(PPE_ROOT / 'images' / 'test' / name), conf=0.3, verbose=False)[0]
        imgs.append(draw_check(r.orig_img.copy(), helmet_check(r))); titles.append(f'{kind}: {name}')  # 판정을 그리고 제목 기록
if imgs:  # 보여 줄 사례가 있을 때만 그림
    show_grid(imgs, titles)

**확인해 보기**
- 놓친 사례(FN)는 왜 생겼나요? 사람이 아예 안 잡혔는지, 모자를 안전모로 착각했는지 구분해 보세요.
- `HEAD_RATIO`, `conf`, 프롬프트(`'helmet'` → `'hard hat'`, `'safety helmet'`)를 바꿔 재현율을 올려 보세요. 안전 시스템에서는 헛경보보다 **놓침이 더 위험**하다는 점을 기억하세요.

결과 파일 (`SAVE_DIR`에 저장)

| 파일 | 내용 |
|---|---|
| `prompt_battle.csv` | 단어별 성적 (`prompt, precision, recall, map50, map50_95`) |
| `model_compare.csv` | 제로샷 vs 학습 모델 (`model, labeled_images, training, map50, ms_per_image`) |
| `helmet_check.csv` | 이미지별 채점 (`image, gt_no_helmet, pred_no_helmet, people_found`) |

---
## 도전 과제

1. **문장형 프롬프트** — `'a person not wearing a helmet'`처럼 문장으로 직접 찾으면 규칙 방식보다 나을까요? Step 5 채점 코드로 비교해 보세요.
2. **조끼까지** — `safety vest`를 추가해 '안전모와 조끼를 모두 착용했는지' 판정하도록 `helmet_check()`를 확장하세요. 데이터셋에는 조끼 미착용 라벨이 없으니 채점 기준을 어떻게 세울지도 생각해 보세요.
3. **보이는 예시로 찾기** — YOLOE는 단어 대신 **예시 박스(visual prompt)** 로도 찾을 수 있습니다. 상자 하나에 박스를 그려 주고 나머지를 찾게 해 보세요. (Ultralytics 문서의 `YOLOEVPSegPredictor` 예제 참고)
4. **다른 데이터로** — 캐글(Kaggle, 공개 데이터셋 공유 사이트)이나 공공데이터에서 받은 사진에 원하는 단어를 넣어 보세요. 어떤 물체는 잘 되고 어떤 물체는 안 되는지 표로 정리합니다.

## 정리

- YOLOE는 **단어로 찾을 대상을 정하는** 탐지 모델입니다. 학습 없이 바로 쓸 수 있습니다.
- 같은 모델이라도 **어떤 단어로 부르느냐**에 따라 성능이 크게 달라집니다. 단어도 데이터로 고릅니다.
- 정확도가 중요한 서비스에서는 아직 **직접 학습한 모델**이 앞섭니다. 제로샷은 빠른 시작과 라벨 초안 만들기에 강합니다.
- 탐지 결과에 **간단한 규칙**을 더하면 '안전모 미착용' 같은 현장 질문에 답할 수 있고, 정답 라벨로 그 규칙을 채점해야 믿고 쓸 수 있습니다.

## 용어 정리

| 용어 | 뜻 |
|---|---|
| YOLOE | 텍스트(단어)나 예시 박스로 찾을 대상을 지정하는 YOLO 계열 탐지·세그멘테이션 모델. 이 실습에서는 yoloe-26s-seg.pt를 쓴다 |
| 오픈 보캐뷸러리(open-vocabulary) 탐지 | 학습 때 정해진 클래스 목록에 묶이지 않고, 사용할 때 주는 단어로 찾을 대상을 정하는 탐지 방식 |
| 제로샷(zero-shot) | 해당 데이터로 추가 학습을 하지 않고 바로 사용하는 것. 이 실습의 YOLOE는 택배 상자 라벨을 한 장도 보지 않았다 |
| 텍스트 프롬프트(prompt) | 모델에게 찾을 대상을 알려 주는 단어나 문장. set_classes()로 지정한다 |
| 텍스트 인코더 (MobileCLIP) | 단어를 숫자 벡터로 바꾸는 신경망. 이 벡터를 이미지 영역 벡터와 비교해 탐지한다 |
| 벡터 유사도 | 두 벡터가 얼마나 같은 방향을 가리키는지의 정도. 단어 벡터와 영역 벡터가 비슷할수록 그 영역을 해당 물체로 판단한다 |
| 백본(backbone) | 이미지에서 특징(영역 벡터)을 뽑아내는 모델의 앞부분 |
| 가중치(weights) | 학습으로 얻은 모델 내부의 숫자 값. .pt 파일에 저장된다 |
| 세그멘테이션(segmentation) | 물체를 박스가 아니라 윤곽(마스크) 단위로 찾는 작업. 모델 이름의 -seg가 이 뜻이다 |
| conf (신뢰도 점수) | 모델이 박스마다 매기는 0~1 점수. predict()의 conf 인자는 이 값 미만의 박스를 버리는 기준(임계값, threshold)이다 |
| IoU (Intersection over Union) | 두 박스가 겹친 넓이 ÷ 합친 넓이. 0.5 이상이면 정답 박스를 맞힌 것으로 본다 |
| mAP50 | IoU 0.5 기준의 평균 정밀도(Average Precision). 점수 순서로 계산하므로 점수 절대값이 낮아도 순서가 맞으면 높게 나온다 |
| mAP50-95 | IoU 기준을 0.5부터 0.95까지 바꿔 가며 구한 AP의 평균. 박스 위치가 얼마나 정확한지까지 반영한다 |
| 정밀도(precision) | 모델이 찾은(경보한) 것 중 맞은 비율. TP/(TP+FP) |
| 재현율(recall) | 실제 정답 중 모델이 찾은 비율. TP/(TP+FN) |
| TP / FN / FP / TN | 잘 찾음 / 놓침 / 헛경보 / 정상을 정상으로 판단. Step 5에서는 이미지 단위로 센다 |
| val() | 검증 데이터로 모델 성능(mAP·정밀도·재현율)을 재는 Ultralytics 함수 |
| batch | 한 번에 묶어 처리하는 이미지 수. 크면 빠르지만 GPU 메모리를 더 쓴다 |
| imgsz | 모델에 넣기 전에 맞추는 이미지 크기(픽셀). 이 실습에서는 640 |
| BGR | OpenCV의 색 순서(파랑·초록·빨강). 화면에 보일 때는 [..., ::-1]로 RGB로 뒤집는다 |
| HEAD_RATIO | 사람 박스 위쪽에서 머리 영역으로 볼 높이 비율. 이 실습에서는 0.35 |
| 비주얼 프롬프트(visual prompt) | 단어 대신 이미지 속 예시 박스로 찾을 대상을 알려 주는 방식 |
| 파인튜닝(fine-tuning) | 사전학습 모델을 내 데이터의 라벨로 추가 학습하는 것. 실습 A의 box_seg_best.pt가 그 결과다 |
| 엣지 장비(edge device) | 서버가 아니라 카메라 옆 현장에 두고 직접 추론하는 소형 컴퓨터. 3회차의 Jetson이 예다 |